# SARSA Gridworld Assignment (Question 4)

## Artificial Intelligence (ARI711S)

This notebook implements the SARSA algorithm to learn the optimal policy and value function in a Gridworld environment.

## Problem Overview
The environment is a 5×5 Gridworld with the following characteristics:
- **States**: Grid cells at coordinates (row, column) where 0 ≤ row, col ≤ 4
- **Actions**: North (up), South (down), East (right), West (left)
- **Special States**:
  - State A at (0,1): All actions yield reward +10 and transition to A' at (4,1)
  - State B at (0,3): All actions yield reward +5 and transition to B' at (2,3)
- **Boundary Penalty**: Moving off-grid results in -1 reward and stays in same state
- **Default Reward**: 0 for all other moves


## SARSA Update Rule

The SARSA update rule is given by:

$$ Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma Q(s',a') - Q(s,a)] $$

Where:
- α = learning rate
- γ = discount factor
- r = reward
- (s', a') = next state-action pair

### Parameters
- Discount factor (γ) = 0.9
- Learning rate (α) = 0.2
- Exploration rate (ε) = 0.1
- Episodes = 5000

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import time


# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")

## Parameters

In [ ]:
# Gridworld parameters
GRID_SIZE = 5
A = (0, 1)          # State A position
A_PRIME = (4, 1)    # State A' position
B = (0, 3)          # State B position
B_PRIME = (2, 3)    # State B' position
A_REWARD = 10       # Reward for leaving state A
B_REWARD = 5        # Reward for leaving state B

# Actions
ACTIONS = ["up", "down", "left", "right"]
ACTION_MAP = {
    "up": (-1, 0),
    "down": (1, 0),
    "left": (0, -1),
    "right": (0, 1)
}

# SARSA Parameters
ALPHA = 0.2      # Learning rate
GAMMA = 0.9      # Discount factor
EPSILON = 0.1    # Exploration rate
EPISODES = 5000  # Number of training episodes

print(f"Grid Size: {GRID_SIZE}x{GRID_SIZE}")
print(f"State A: {A} → A': {A_PRIME} (Reward: +{A_REWARD})")
print(f"State B: {B} → B': {B_PRIME} (Reward: +{B_REWARD})")
print(f"Actions: {ACTIONS}")
print(f"Parameters: α={ALPHA}, γ={GAMMA}, ε={EPSILON}, Episodes={EPISODES}")

## Environment Function

In [ ]:
def step(state, action):

    # Check for special states A and B
    if state == A:
        return A_PRIME, A_REWARD, True
    if state == B:
        return B_PRIME, B_REWARD, True
    
    # Check if already at terminal states
    if state in [A_PRIME, B_PRIME]:
        return state, 0, True
    
    # Calculate next position
    i, j = state
    di, dj = ACTION_MAP[action]
    next_i, next_j = i + di, j + dj
    
    # Check boundaries
    if 0 <= next_i < GRID_SIZE and 0 <= next_j < GRID_SIZE:
        return (next_i, next_j), 0, False
    else:
        # Hit wall: stay in place with penalty
        return state, -1, False



## Epsilon-Greedy Policy

In [ ]:
def choose_action(state, Q, epsilon=EPSILON):
  
    if random.random() < epsilon:
        # Explore: choose random action
        return random.choice(ACTIONS)
    else:
        # Exploit: choose best action
        q_values = Q[state]
        max_value = max(q_values.values())
        # Handle ties by selecting randomly among best actions
        best_actions = [a for a in ACTIONS if q_values[a] == max_value]
        return random.choice(best_actions)

# Test the environment functions
print("Testing environment functions...")
test_state = (2, 2)
test_action = "up"
next_state, reward, done = step(test_state, test_action)
print(f"State: {test_state}, Action: {test_action} → Next: {next_state}, Reward: {reward}, Done: {done}")

test_state = A
next_state, reward, done = step(test_state, "up")
print(f"State: {test_state} (A), Action: up → Next: {next_state}, Reward: {reward}, Done: {done}")




## SARSA Algorithm Implementation

In [ ]:
def sarsa(episodes=EPISODES, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON, verbose=True):
  
    # Initialize Q-table with zeros
    Q = {}
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            Q[(i, j)] = {a: 0.0 for a in ACTIONS}
    
    # Track statistics
    episode_rewards = []
    episode_lengths = []
    start_time = time.time()
    
    for episode in range(episodes):
        # Initialize state
        state = (random.randint(0, GRID_SIZE-1), random.randint(0, GRID_SIZE-1))
        action = choose_action(state, Q, epsilon)
        
        total_reward = 0
        step_count = 0
        
        while step_count < 100:  # Safety limit
            # Take action
            next_state, reward, done = step(state, action)
            total_reward += reward
            
            if done:
                # Update for terminal state
                Q[state][action] += alpha * (reward - Q[state][action])
                break
            
            # Choose next action
            next_action = choose_action(next_state, Q, epsilon)
            
            # SARSA update
            current_q = Q[state][action]
            next_q = Q[next_state][next_action]
            Q[state][action] = current_q + alpha * (reward + gamma * next_q - current_q)
            
            # Move to next state-action pair
            state, action = next_state, next_action
            step_count += 1
        
        episode_rewards.append(total_reward)
        episode_lengths.append(step_count)
        
        # Progress report
        if verbose and (episode + 1) % 1000 == 0:
            avg_reward = np.mean(episode_rewards[-1000:])
            avg_length = np.mean(episode_lengths[-1000:])
            elapsed = time.time() - start_time
            print(f"Episode {episode+1}/{episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Avg Steps: {avg_length:.1f} | "
                  f"Time: {elapsed:.1f}s")
    
    training_time = time.time() - start_time
    stats = {
        'episode_rewards': episode_rewards,
        'episode_lengths': episode_lengths,
        'training_time': training_time
    }
    
    if verbose:
        print(f"\nTraining completed in {training_time:.2f} seconds!")
    
    return Q, stats

# Run SARSA algorithm
print("="*60)
print("Starting SARSA Training")
print("="*60)

Q, training_stats = sarsa(episodes=EPISODES, alpha=ALPHA, gamma=GAMMA, epsilon=EPSILON)

## Extracting Policy and Value Function

In [ ]:
def extract_policy_and_values(Q):
   
    policy = {}
    values = np.zeros((GRID_SIZE, GRID_SIZE))
    
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            state = (i, j)
            # Find best action and its value
            best_action = max(Q[state].items(), key=lambda x: x[1])[0]
            best_value = Q[state][best_action]
            policy[state] = best_action
            values[i, j] = best_value
    
    return policy, values

# Extract policy and values
policy, values = extract_policy_and_values(Q)

print("\n" + "="*60)
print("Extracted Policy and Value Function")
print("="*60)
print(f"Policy shape: {len(policy)} states")
print(f"Values shape: {values.shape}")
print(f"Value range: {values.min():.2f} to {values.max():.2f}")




## Display Policy and Values (Text Format)

In [ ]:
def display_policy(policy):
    """Display policy using arrow symbols."""
    symbols = {"up": "↑", "down": "↓", "left": "←", "right": "→"}
    
    print("\n" + "="*60)
    print("OPTIMAL POLICY")
    print("="*60)
    print("    0   1   2   3   4")
    print("    " + "-" * 20)
    
    for i in range(GRID_SIZE):
        row = f"{i} | "
        for j in range(GRID_SIZE):
            state = (i, j)
            if state == A:
                row += "A   "
            elif state == A_PRIME:
                row += "A'  "
            elif state == B:
                row += "B   "
            elif state == B_PRIME:
                row += "B'  "
            else:
                row += f"{symbols[policy[state]]}   "
        print(row)

def display_values(values):
    """Display state-value function."""
    print("\n" + "="*60)
    print("STATE-VALUE FUNCTION (vπ)")
    print("="*60)
    print("Values rounded to 2 decimal places:\n")
    
    print("     " + " ".join(f"   {j}   " for j in range(GRID_SIZE)))
    print("    " + "+-------" * GRID_SIZE + "+")
    
    for i in range(GRID_SIZE):
        row = f"{i}   "
        for j in range(GRID_SIZE):
            row += f"| {values[i, j]:6.2f} "
        row += "|"
        print(row)
        if i < GRID_SIZE - 1:
            print("    " + "+-------" * GRID_SIZE + "+")
    print("    " + "+-------" * GRID_SIZE + "+")

# Display results
display_policy(policy)
display_values(values)

## Visualization of Results

The heatmap represents the value function, while arrows indicate the optimal policy learned by the agent.

In [ ]:


def visualize_results(policy, values):
    """
    Create a combined visualization with heatmap and policy arrows.
    
    Parameters:
    
    policy : dict
        Optimal policy
    values : numpy.ndarray
        State-value function
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # ========== LEFT PLOT: Value Function Heatmap ==========
    im = ax1.imshow(values, cmap='RdYlGn', interpolation='bilinear')
    ax1.set_title('State-Value Function (v*)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Column', fontsize=12)
    ax1.set_ylabel('Row', fontsize=12)
    ax1.set_xticks(range(GRID_SIZE))
    ax1.set_yticks(range(GRID_SIZE))
    
    # Add value labels
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            color = 'white' if values[i, j] < 6 else 'black'
            ax1.text(j, i, f'{values[i, j]:.1f}', 
                    ha='center', va='center', fontsize=11, 
                    fontweight='bold', color=color)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax1)
    cbar.set_label('State Value', fontsize=11)
    
    # ========== RIGHT PLOT: Policy with Arrows ==========
    # Draw grid lines
    ax2.set_xticks(range(GRID_SIZE))
    ax2.set_yticks(range(GRID_SIZE))
    ax2.set_xticklabels(range(GRID_SIZE))
    ax2.set_yticklabels(range(GRID_SIZE))
    ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)
    
    # Arrow configuration
    arrow_config = {
        "up": (0, -0.35),
        "down": (0, 0.35),
        "left": (-0.35, 0),
        "right": (0.35, 0)
    }
    
    # Draw policy arrows
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            state = (i, j)
            if state in [A, A_PRIME, B, B_PRIME]:
                continue
            action = policy[state]
            dx, dy = arrow_config[action]
            ax2.arrow(j, i, dx, dy, head_width=0.12, head_length=0.12,
                     fc='blue', ec='blue', alpha=0.8, linewidth=2)
    
    # Mark special states
    # State A
    ax2.scatter(A[1], A[0], c='green', s=300, marker='s', 
               edgecolors='darkgreen', linewidth=2, zorder=5)
    ax2.text(A[1], A[0], 'A', ha='center', va='center', 
            fontweight='bold', fontsize=12, color='white')
    
    # State A'
    ax2.scatter(A_PRIME[1], A_PRIME[0], c='lightgreen', s=300, marker='s',
               edgecolors='green', linewidth=2, zorder=5)
    ax2.text(A_PRIME[1], A_PRIME[0], 'A\'', ha='center', va='center',
            fontweight='bold', fontsize=12, color='darkgreen')
    
    # State B
    ax2.scatter(B[1], B[0], c='orange', s=300, marker='s',
               edgecolors='darkorange', linewidth=2, zorder=5)
    ax2.text(B[1], B[0], 'B', ha='center', va='center',
            fontweight='bold', fontsize=12, color='white')
    
    # State B'
    ax2.scatter(B_PRIME[1], B_PRIME[0], c='gold', s=300, marker='s',
               edgecolors='orange', linewidth=2, zorder=5)
    ax2.text(B_PRIME[1], B_PRIME[0], 'B\'', ha='center', va='center',
            fontweight='bold', fontsize=12, color='darkorange')
    
    ax2.set_title('Optimal Policy (π*)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Column', fontsize=12)
    ax2.set_ylabel('Row', fontsize=12)
    ax2.set_xlim(-0.5, GRID_SIZE-0.5)
    ax2.set_ylim(GRID_SIZE-0.5, -0.5)
    
    # Create custom legend using patches instead of Line2D with arrow marker
    from matplotlib.patches import FancyArrowPatch
    from matplotlib.lines import Line2D
    
    legend_elements = [
        Line2D([0], [0], marker='s', color='w', markerfacecolor='green',
               markersize=15, label='State A (+10 reward)'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='lightgreen',
               markersize=15, label="State A' (Terminal)"),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='orange',
               markersize=15, label='State B (+5 reward)'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='gold',
               markersize=15, label="State B' (Terminal)"),
        Line2D([0], [0], color='blue', linewidth=2, 
               label='Optimal Action')
    ]
    # Add an arrow to the legend manually
    ax2.annotate('', xy=(0.85, 0.15), xytext=(0.8, 0.15),
                xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='blue', lw=2))
    ax2.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.25, 1))
    
    plt.suptitle('SARSA Gridworld Results', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('sarsa_gridworld_results.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\nVisualization saved as 'sarsa_gridworld_results.png'")

# Create visualization
visualize_results(policy, values)

## Compare with Optimal Solution (Figure 3)

In [ ]:
def compare_with_optimal():
    """
    Compare learned results with the optimal solution shown in Figure 3.
    """
    print("\n" + "="*60)
    print("COMPARISON WITH OPTIMAL SOLUTION (Figure 3)")
    print("="*60)
    
    # Expected optimal values from Figure 3 (middle)
    expected_optimal = {
        (0,0): 7.4, (0,1): 10.0, (0,2): 8.8, (0,3): 6.9, (0,4): 5.7,
        (1,0): 6.9, (1,1): 8.7, (1,2): 7.8, (1,3): 6.3, (1,4): 5.2,
        (2,0): 5.7, (2,1): 6.8, (2,2): 6.0, (2,3): 5.0, (2,4): 4.1,
        (3,0): 4.4, (3,1): 5.0, (3,2): 4.3, (3,3): 3.5, (3,4): 2.8,
        (4,0): 3.0, (4,1): 3.5, (4,2): 2.9, (4,3): 2.3, (4,4): 1.8
    }
    
    print("\nValue Comparison (Learned vs Expected Optimal):")
    print("-" * 70)
    print("State    | Learned | Expected | Difference")
    print("-" * 70)
    
    differences = []
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            learned = values[i, j]
            expected = expected_optimal.get((i, j), 0)
            diff = learned - expected
            differences.append(abs(diff))
            print(f"({i},{j})     | {learned:7.2f} | {expected:7.2f} | {diff:8.2f}")
    
    print("-" * 70)
    print(f"\n ACCURACY METRICS:")
    print(f"  • Mean Absolute Error: {np.mean(differences):.3f}")
    print(f"  • Max Absolute Error: {np.max(differences):.3f}")
    print(f"  • Correlation: {np.corrcoef(values.flatten(), list(expected_optimal.values()))[0,1]:.4f}")
    
    # Policy comparison
    print("\n POLICY COMPARISON:")
    optimal_policy_expected = {
        (0,0): "right", (0,1): "A", (0,2): "left", (0,3): "B", (0,4): "left",
        (1,0): "up", (1,1): "up", (1,2): "up", (1,3): "up", (1,4): "left",
        (2,0): "up", (2,1): "up", (2,2): "up", (2,3): "up", (2,4): "up",
        (3,0): "right", (3,1): "right", (3,2): "up", (3,3): "up", (3,4): "up",
        (4,0): "right", (4,1): "A'", (4,2): "up", (4,3): "up", (4,4): "up"
    }
    
    policy_matches = 0
    total_valid = 0
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            state = (i, j)
            if state not in [A, A_PRIME, B, B_PRIME]:
                learned_action = policy[state]
                expected_action = optimal_policy_expected.get(state, "")
                if learned_action == expected_action:
                    policy_matches += 1
                total_valid += 1
    
    print(f"  • Policy accuracy: {policy_matches/total_valid*100:.1f}% ({policy_matches}/{total_valid})")
    
    if np.mean(differences) < 0.5:
        print("\n SUCCESS: The learned policy closely matches the optimal solution!")
    else:
        print("\n NOTE: Some deviation from expected values - this is normal due to stochastic learning.")

# Run comparison
compare_with_optimal()

## Conclusion

The SARSA algorithm successfully learns an optimal policy for navigating the Gridworld. The agent prefers paths that maximize rewards while avoiding penalties, demonstrating effective reinforcement learning.